# Latent Steps vs CoT Hint — Replication Experiment

**Goal:** Replicate the `decoded_latent.txt` run (2 latent steps + truncated CoT hint),
then sweep latent step counts on the 149 "original-only" questions to study how latent
depth affects accuracy when explicit CoT hints are removed.

---

**Runtime recommendation:** GPU → T4 (Runtime → Change runtime type → GPU → T4)

| Hardware | Full dataset (~1319 q, batch=32) | 149-q subset | Notes |
|----------|----------------------------------|--------------|-------|
| CPU only | ~8–14 hours | ~60–90 min | ❌ Not recommended |
| **T4 GPU** | **~25–40 min** | **~3–5 min** | ✅ Free Colab tier |
| L4 GPU | ~12–20 min | ~1–2 min | Colab Pro |
| A100 GPU | ~6–10 min | ~30–60 sec | Colab Pro+ |

> Each question requires ~260 forward passes (7 latent + up to 256 decode steps).
> CPU forward pass ≈ 200–400ms vs ≈ 5ms on T4 — GPU is non-negotiable for any real run.


## 1 — Install dependencies

In [ ]:
%%capture
!pip install \
    transformers==4.40.0 \
    peft==0.10.0 \
    datasets==2.18.0 \
    accelerate==0.29.3 \
    safetensors \
    bitsandbytes
print("Install complete.")


## 2 — Clone repository

In [ ]:
import os

REPO = "LatentModelPositionalAnalysis"
if not os.path.exists(REPO):
    !git clone https://github.com/Hieuuum/LatentModelPositionalAnalysis

%cd {REPO}
!ls


## 3 — Mount Google Drive (checkpoint)

Your checkpoint must be in Google Drive. Expected path: `MyDrive/codi_gpt2/model.safetensors`
(or `pytorch_model.bin`). Adjust `CKPT_DIR` below if your path differs.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

CKPT_DIR = "/content/drive/MyDrive/codi_gpt2"   # ← adjust if needed
print(f"Checkpoint dir: {CKPT_DIR}")
assert os.path.isdir(CKPT_DIR), f"Not found: {CKPT_DIR}. Check your Drive path."
print("✓ Checkpoint directory found.")


## 4 — Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU: {name}  |  VRAM: {vram:.1f} GB")
else:
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → GPU → T4")


## 5 — Configuration

Edit these variables to control what gets run.


In [ ]:
import os, json

# ── Latent sweep values for Phase 2 ────────────────────────────────────────
LATENT_SWEEP = [2, 4, 6, 8, 10]

# ── Phase 1 replication: full dataset (False) or subset only (True) ─────────
RUN_SUBSET_ONLY = False   # keep False for Phase 1 replication

# ── 149 "original-only" question indices ────────────────────────────────────
# These are questions correct with 6-latent no-hint, wrong with 2-latent CoT-hint
# Source: outputs/decoded_latent_comparison.txt
ORIGINAL_ONLY_INDICES = [
    4, 12, 20, 30, 31, 33, 55, 61, 66, 111, 114, 131, 159, 166, 178, 183, 187,
    222, 224, 227, 230, 248, 266, 276, 278, 285, 298, 299, 315, 323, 330, 334,
    338, 346, 347, 348, 361, 362, 371, 373, 375, 380, 381, 382, 392, 396, 397,
    403, 415, 426, 428, 450, 476, 478, 486, 488, 530, 539, 553, 574, 578, 584,
    598, 602, 624, 630, 636, 647, 648, 655, 659, 671, 673, 676, 691, 692, 698,
    702, 723, 725, 740, 742, 755, 761, 772, 773, 776, 788, 789, 798, 804, 816,
    818, 824, 829, 832, 836, 861, 873, 885, 894, 908, 909, 911, 925, 932, 933,
    936, 942, 946, 957, 970, 980, 988, 989, 1007, 1009, 1013, 1015, 1029, 1031,
    1050, 1060, 1063, 1069, 1075, 1084, 1096, 1103, 1105, 1106, 1108, 1110,
    1119, 1120, 1121, 1126, 1151, 1157, 1160, 1166, 1172, 1175, 1184, 1185,
    1186, 1192, 1207, 1234
]

os.makedirs("outputs", exist_ok=True)
with open("outputs/original_only_indices.txt", "w") as f:
    f.write("\n".join(str(i) for i in ORIGINAL_ONLY_INDICES))

print(f"Subset size:   {len(ORIGINAL_ONLY_INDICES)} questions")
print(f"Latent sweep:  {LATENT_SWEEP}")
print(f"Subset-only mode: {RUN_SUBSET_ONLY}")


## 6 — Run helper

`run_probe()` launches either `probe_latent_cot_hint.py` (with CoT hint) or
`probe_latent_token.py` (no hint) with the given config.


In [ ]:
import os, sys

def run_probe(latent_iterations, use_hint=True, question_indices_file=None):
    """Build argument list and run script via IPython ! (streams output to cell)."""
    script = "probe_latent_cot_hint.py" if use_hint else "probe_latent_token.py"

    args = [
        f"--data_name zen-E/GSM8k-Aug",
        f"--output_dir outputs",
        f"--model_name_or_path gpt2",
        f"--seed 11",
        f"--model_max_length 512",
        f"--bf16",
        f"--lora_r 128 --lora_alpha 32 --lora_init",
        f"--batch_size 32",
        f"--greedy True",
        f"--num_latent 6",
        f"--use_prj True",
        f"--prj_dim 768",
        f"--prj_no_ln False",
        f"--prj_dropout 0.0",
        f"--inf_latent_iterations {latent_iterations}",
        f"--inf_num_iterations 1",
        f"--remove_eos True",
        f"--use_lora True",
        f"--ckpt_dir {CKPT_DIR}",
    ]
    if question_indices_file:
        args.append(f"--question_indices {question_indices_file}")

    cmd = f"python {script} " + " ".join(args)

    hint_str   = "CoT-hint" if use_hint else "no-hint"
    subset_str = f"subset({len(ORIGINAL_ONLY_INDICES)}q)" if question_indices_file else "full-dataset"
    print(f"\n{'='*60}")
    print(f"  latent={latent_iterations} | {hint_str} | {subset_str}")
    print(f"  CWD: {os.getcwd()}")
    print(f"{'='*60}\n")
    print(f"CMD: {cmd}\n")

    return cmd   # caller uses ! to run it

print("run_probe() ready  —  returns command string for ! invocation")
print(f"Python: {sys.executable}")
print(f"CWD:    {os.getcwd()}")


## 7 — Phase 1: Replication Run

Full dataset · 2 latent steps · **with CoT hint**

Expected outcome: ~544 correct (matching original `decoded_latent.txt`).


In [ ]:
import os, sys

# ── Preflight ────────────────────────────────────────────────────────────────
print("Preflight checks:")
checks = {
    "probe_latent_cot_hint.py": os.path.exists("probe_latent_cot_hint.py"),
    "src/model.py":             os.path.exists("src/model.py"),
    "outputs/ dir":             os.path.isdir("outputs"),
    "CKPT_DIR":                 os.path.isdir(CKPT_DIR),
}
os.makedirs("outputs", exist_ok=True)
for k, v in checks.items():
    print(f"  {'✅' if v else '❌'}  {k}")

if not all(checks.values()):
    print("\n❌  Fix preflight failures above before running.")
else:
    print("\n✅  All checks passed. Running --help to validate args...")

    # Use get_ipython().system() which streams output into the cell
    ip = get_ipython()

    # Test argument parsing first
    ip.system(f"python probe_latent_cot_hint.py --help 2>&1 | head -20")
    print("\n(If help text appeared above, args are fine.)")
    print("\nStarting Phase 1: 2 latents, CoT hint, full dataset (~30 min on T4)...")

    # Build command string and run with ! via IPython
    script_args = (
        "probe_latent_cot_hint.py"
        " --data_name zen-E/GSM8k-Aug"
        " --output_dir outputs"
        " --model_name_or_path gpt2"
        " --seed 11"
        " --model_max_length 512"
        " --bf16"
        " --lora_r 128 --lora_alpha 32 --lora_init"
        " --batch_size 32"
        " --greedy True"
        " --num_latent 6"
        " --use_prj True"
        " --prj_dim 768"
        " --prj_no_ln False"
        " --prj_dropout 0.0"
        " --inf_latent_iterations 2"
        " --inf_num_iterations 1"
        " --remove_eos True"
        " --use_lora True"
        f" --ckpt_dir {CKPT_DIR}"
    )
    ip.system(f"python {script_args} 2>&1")


## 8 — Verify replication result

Load the JSON summary and confirm ~544 correct answers.


In [ ]:
import json

with open("outputs/decoded_latent_cot_hint_latent2.json") as f:
    data = json.load(f)

print(f"Accuracy:     {data['accuracy']:.2f}%")
print(f"Correct:      {data['num_correct']} / {data['config']['num_questions']}")
print(f"Latent steps: {data['config']['inf_latent_iterations']}")

correct_indices = [r['question_idx'] for r in data['results'] if r['correct']]
print(f"\nFirst 20 correct Q indices: {correct_indices[:20]}")

# How many of the 149 original-only questions did the hint model get right?
overlap = set(correct_indices) & set(ORIGINAL_ONLY_INDICES)
print(f"\nOf 149 'original-only' questions, hint model got {len(overlap)} correct")
print("(Expect 0 — these are questions where the hint HURT the model)")


## 9 — Phase 2: Baseline regeneration + Latent Sweep

First run 6-latent no-hint on full dataset to regenerate `original_decoded_latent`,
then sweep latent counts on the 149-question subset.


In [ ]:
# Step 1: Regenerate original baseline (6 latents, no hint, full dataset)
print("Step 1: Regenerating original baseline (6 latents, no hint)...")
rc = run_probe(latent_iterations=6, use_hint=False, question_indices_file=None)
print(f"Baseline done (rc={rc})\n")


In [ ]:
# Step 2: Sweep latent counts on 149-question subset (no hint)
sweep_accuracies = {}

for n_latent in LATENT_SWEEP:
    indices_file = "outputs/original_only_indices.txt" if RUN_SUBSET_ONLY else None
    rc = run_probe(
        latent_iterations=n_latent,
        use_hint=False,
        question_indices_file=indices_file,
    )
    suffix = f"_latent{n_latent}"
    if RUN_SUBSET_ONLY:
        suffix += f"_subset{len(ORIGINAL_ONLY_INDICES)}q"
    json_path = f"outputs/decoded_latent_cot_hint{suffix}.json"
    if os.path.exists(json_path):
        with open(json_path) as f:
            d = json.load(f)
        sweep_accuracies[n_latent] = d['accuracy']
        print(f"  latent={n_latent}: {d['accuracy']:.2f}% ({d['num_correct']}/{d['config']['num_questions']} correct)")
    else:
        print(f"  latent={n_latent}: output JSON not found (check script output above)")


## 10 — Plot accuracy curve

In [ ]:
import matplotlib.pyplot as plt

if sweep_accuracies:
    xs = sorted(sweep_accuracies.keys())
    ys = [sweep_accuracies[x] for x in xs]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(xs, ys, marker='o', linewidth=2.5, color='steelblue', markersize=8)
    for x, y in zip(xs, ys):
        ax.annotate(f"{y:.1f}%", (x, y), textcoords="offset points",
                    xytext=(0, 10), ha='center', fontsize=10)
    ax.set_xlabel("Latent Steps (inf_latent_iterations)", fontsize=12)
    ax.set_ylabel("Accuracy on 149-question subset (%)", fontsize=12)
    ax.set_title(
        "Effect of Latent Step Count on Questions\n"
        "That Require Pure Latent Reasoning (No CoT Hint)",
        fontsize=13
    )
    ax.set_xticks(xs)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig("outputs/latent_sweep_accuracy.png", dpi=150)
    plt.show()
    print("Saved: outputs/latent_sweep_accuracy.png")
else:
    print("No sweep results found. Run Cell 9 first.")


## 11 — Download results

In [ ]:
from google.colab import files
import glob, os

output_files = sorted(
    glob.glob("outputs/*.json") +
    glob.glob("outputs/*.txt") +
    glob.glob("outputs/*.png")
)

print("Available output files:")
for f in output_files:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {f:<55} {size_kb:>7.1f} KB")

print("\nUncomment below to download:")
print("# for f in output_files: files.download(f)")

# Uncomment to download all:
# for f in output_files:
#     files.download(f)
